# Mining project

## Import libries and the file

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [6]:
df = pd.read_csv("data/mining_block_model.csv")

## Search DataFrame

### Обозначения

1. Block_ID: Уникальный идентификатор для блока.

2. Spatial Coordinates (X, Y, Z): координаты блока.

3. Rock Type: тип породы (e.g., Hematite, Magnetite, Waste).

4. Ore Grade (%): содержание полезного ископаемого.

5. Tonnage (tonnes): вес блока в тоннах.

6. Ore Value (¥/tonne): ценность руды на тонну из блока.

7. Mining Cost (¥): затраты на систему вскрытия и добычи.

8. Processing Cost (¥): затраты на обогатительную фабрику.

9. Waste Flag: флаг куба (1 - вскрыша, 0 - руда)

10. Profit (¥): финансовый итог куба (Доход от продажи руды - Стоимость добычи - Стоимость переработки).

11. Target: выгодно ли добывать блок (1 = Yes, 0 = No).

### DataFrame

In [9]:
df

,Block_ID,X,Y,Z,Rock_Type,Ore_Grade (%),Tonnage,Ore_Value (¥/tonne),Mining_Cost (¥),Processing_Cost (¥),Waste_Flag,Profit (¥),Target
0,B00001,102,186,6,Magnetite,51.93,2131,294.48,53,38,0,433615.88,1
1,B00002,435,448,82,Hematite,59.05,1550,273.00,36,33,0,316200.00,1
2,B00003,348,476,94,Magnetite,63.79,2414,338.36,57,28,0,611611.04,1
3,B00004,270,127,98,Hematite,64.98,1297,307.60,30,29,0,322434.20,1
4,B00005,106,111,92,Waste,0.00,1309,0.00,67,28,1,-124355.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
74995,B74996,352,111,18,Magnetite,52.05,1582,295.84,33,38,0,355696.88,1
74996,B74997,373,424,33,Hematite,57.25,1735,317.43,55,24,0,413676.05,1
74997,B74998,23,261,7,Hematite,59.10,2294,340.13,45,34,0,599032.22,1
74998,B74999,305,109,95,Waste,0.00,2639,0.00,63,27,1,-237510.00,0


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Block_ID             75000 non-null  str    
 1   X                    75000 non-null  int64  
 2   Y                    75000 non-null  int64  
 3   Z                    75000 non-null  int64  
 4   Rock_Type            75000 non-null  str    
 5   Ore_Grade (%)        75000 non-null  float64
 6   Tonnage              75000 non-null  int64  
 7   Ore_Value (¥/tonne)  75000 non-null  float64
 8   Mining_Cost (¥)      75000 non-null  int64  
 9   Processing_Cost (¥)  75000 non-null  int64  
 10  Waste_Flag           75000 non-null  int64  
 11  Profit (¥)           75000 non-null  float64
 12  Target               75000 non-null  int64  
dtypes: float64(3), int64(8), str(2)
memory usage: 7.4 MB


In [10]:
df.columns

Index(['Block_ID', 'X', 'Y', 'Z', 'Rock_Type', 'Ore_Grade (%)', 'Tonnage',
       'Ore_Value (¥/tonne)', 'Mining_Cost (¥)', 'Processing_Cost (¥)',
       'Waste_Flag', 'Profit (¥)', 'Target'],
      dtype='str')

In [16]:
df.corr(numeric_only=True)["Target"][:-1]

X                     -0.003162
Y                     -0.004109
Z                     -0.000270
Ore_Grade (%)          0.986047
Tonnage                0.003775
Ore_Value (¥/tonne)    0.977663
Mining_Cost (¥)       -0.004308
Processing_Cost (¥)   -0.002441
Waste_Flag            -1.000000
Profit (¥)             0.877871
Name: Target, dtype: float64

In [17]:
df.describe()

,X,Y,Z,Ore_Grade (%),Tonnage,Ore_Value (¥/tonne),Mining_Cost (¥),Processing_Cost (¥),Waste_Flag,Profit (¥),Target
count,75000.000000,75000.000000,75000.000000,75000.000000,75000.000000,75000.000000,75000.000000,75000.000000,75000.000000,75000.000000,75000.000000
mean,249.242680,248.773240,49.525360,45.992171,2000.475080,239.848683,49.563547,29.512120,0.200293,322005.651984,0.799707
std,144.613539,143.852756,28.858031,23.343018,577.028563,122.777574,11.526363,5.772658,0.400223,273713.865800,0.400223
min,0.000000,0.000000,0.000000,0.000000,1000.000000,0.000000,30.000000,20.000000,0.000000,-317152.000000,0.000000
25%,123.000000,124.000000,24.000000,50.920000,1501.000000,256.170000,40.000000,24.000000,0.000000,238527.937500,1.000000
50%,249.000000,248.000000,50.000000,55.620000,2003.000000,287.290000,50.000000,29.000000,0.000000,378189.050000,1.000000
75%,375.000000,373.000000,75.000000,60.340000,2501.000000,318.610000,60.000000,35.000000,0.000000,516542.122500,1.000000
max,499.000000,499.000000,99.000000,65.000000,2999.000000,350.000000,69.000000,39.000000,1.000000,883522.030000,1.000000


### Вывод

Линейной связи между координатами и таргетной переменной нет, присутствует сильная корреляция с признаками `Ore_Grade (%)` и `Ore_Value (¥/tonne)`

Признак `Waste_Flag` обратно коррелирует с таргетной переменной, поэтому его нужно удалить.